## Import the libraries

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents import Agent
from agents import Runner

import requests
from agents import function_tool

## Load the environment variables

In [2]:
load_dotenv()

True

## Check if the variables are loaded 

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
serper_api_key = os.getenv('SERPER_API_KEY')

In [4]:
print(openai_api_key[:2])

sk


In [5]:
print(serper_api_key[:2])

e5


## Call the api from agent

In [12]:
@function_tool
def web_search(query: str) -> str:

    url = "https://google.serper.dev/search"

    payload = {
        "q": query
    }

    headers = {
        "X-API-KEY": serper_api_key,
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        json=payload,
        headers=headers
    )

    response.raise_for_status()

    data = response.json()

    organic_results = data.get("organic", [])

    if not organic_results:
        return "No results found."

    formatted_results = []

    for item in organic_results[:5]:

        formatted_results.append(
            f"""
Title: {item.get('title')}
Snippet: {item.get('snippet')}
Link: {item.get('link')}
"""
        )

    return "\n".join(formatted_results)

## Defining the agent

In [13]:
instructions = """
You are a research assistant.

For questions requiring current or factual web information,
ALWAYS use the web_search tool first before answering.

Use the search results to generate accurate, detailed responses.
"""

In [14]:
agent = Agent(
    name = "customized web search",
    instructions = instructions,
    model = 'gpt-4o-mini',
    tools = [web_search]
)


In [15]:
response = await Runner.run(
    starting_agent = agent,
    input = "What are the top Python frameworks for building AI Agents? Compare their features, advantages, and limitations"
)
    

In [16]:
print(response)

RunResult:
- Last agent: Agent(name="customized web search", ...)
- Final output (str):
    Below is a comparison of several prominent Python frameworks for building AI agents, focusing on their features, advantages, and limitations.
    
    ### 1. LangGraph
    
    **Features:**
    - Graph-based workflows and state management
    - Designed for complex generative AI applications
    - Integration with various data sources and language models
    
    **Advantages:**
    - Excellent for applications requiring sophisticated logic and memory persistence
    - Promotes control and transparency in workflows, making it easier to manage complex interdependencies
    - Open-source with a supportive community
    
    **Limitations:**
    - Can be seen as less intuitive for simple tasks compared to more straightforward frameworks
    - May lack some advanced features that are built into more specialized frameworks
    
    ### 2. CrewAI
    
    **Features:**
    - Designed for multi-agent 